## Agent Traces to Supervised Fine-tuning

This demo showcases how Agent Traces stored in App Insights can be used to perform Supervised Fine-tuning (SFT) on an AzureOpenAI model.

### Prerequisites

1. You'll need `Owner` or `RBAC Administrator` role on your Azure subscription to assign roles.
2. You'll need to Deploy the Hosted Agent to your Foundry project by following the README in this directory: [retail-agent-langgraph](./retail-agent-langgraph).

After deploying the agent in your Foundry project, confirm that there are enough agent traces to perform fine-tuning. In Foundry, navigate to `Agents > {your-agent} > Traces > Conversations`. There should be at least 10 conversations to proceed.


### Setup .env file

Copy the .env.template file to .env and update the placeholders with your Foundry project and App Insights details.

In [ ]:
!cp .env.template .env

### Install Python Packages

Restart your kernel after running the below pip install command.

In [ ]:
%pip install azure_ai_projects-2.2.0-py3-none-any.whl azure-mgmt-cognitiveservices dotenv

### Grant Foundry project access to App Insights

Foundry project service principal requires `Log Analytics Reader` role on the App Insights resource to read the agent traces. Grant the access by following these steps: 

In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

FOUNDRY_PROJECT_ID = os.environ["FOUNDRY_PROJECT_ID"]
APPLICATION_INSIGHTS_ID = os.environ["APPLICATION_INSIGHTS_ID"]

Fetch Foundry Project System-Assigned Managed Identity's Principal ID

In [ ]:
!az resource show --ids {FOUNDRY_PROJECT_ID} --query identity.principalId -o tsv

Copy the above Principal ID in the below command:

In [ ]:
!az role assignment create --assignee "COPIED_PRINCIPAL_ID" --role "Log Analytics Reader" --scope {APPLICATION_INSIGHTS_ID}

### Start Data Generation Job

Start Data Generation Job to export Agent Traces in App Insights to a Supervised Fine-tuning file.

In [7]:
from azure.ai.projects import AIProjectClient
from azure.identity import DefaultAzureCredential

credential = DefaultAzureCredential()
project_client = AIProjectClient(
    endpoint=os.environ["FOUNDRY_PROJECT_ENDPOINT"],
    credential=credential
)

In [ ]:
from datetime import datetime, timedelta, timezone

from azure.ai.projects.models import (
    DataGenerationJob,
    DataGenerationJobInputs,
    DataGenerationJobScenario,
    TracesDataGenerationJobOptions,
    TracesDataGenerationJobSource,
)

AGENT_NAME = "retail-agent-langgraph"
TRACES_START_TIME = datetime.now(timezone.utc) - timedelta(days=1)

options = TracesDataGenerationJobOptions(
    max_samples=50,  # maxinum number of samples in output files
    train_split=0.8  # split output into training and validation files
)

source = TracesDataGenerationJobSource(
    agent_name=AGENT_NAME,
    start_time=int(TRACES_START_TIME.timestamp())
)

def _put_type_first(model):
    # TODO: Fix API bug to avoid this SDK workaround.
    if hasattr(model, "_data") and "type" in model._data:
        model._data = {"type": model._data["type"], **{k: v for k, v in model._data.items() if k != "type"}}
_put_type_first(options)
_put_type_first(source)

job = DataGenerationJob(
    inputs=DataGenerationJobInputs(
        name="agent_traces_to_sft",
        scenario=DataGenerationJobScenario.SUPERVISED_FINETUNING,
        options=options,
        sources=[source]
    )
)

job = project_client.beta.datasets.create_generation_job(job)
print(f"Data generation job created: {job.id}")


Data generation job created: datagen-b10b86ed457a4924b3f19e7f6b40cd1e


In [11]:
import time

from azure.ai.projects.models import JobStatus
from IPython.display import clear_output

while job.status not in [JobStatus.SUCCEEDED, JobStatus.FAILED, JobStatus.CANCELLED]:
    job = project_client.beta.datasets.get_generation_job(job.id)
    clear_output(wait=True)
    print(f"Job status: {job.status}")
    time.sleep(10)

print(f"Data generation job finished with status: {job.status}")
if job.status == JobStatus.FAILED:
    raise Exception("Data generation job failed with error:", job.error)

Job status: JobStatus.SUCCEEDED
Data generation job finished with status: JobStatus.SUCCEEDED


In [12]:
from azure.ai.projects.models import DataGenerationJobOutputType

for output in job.result.outputs:
    assert output.type == DataGenerationJobOutputType.FILE

    print(f"Output {output.type}: File ID: {output.id}, Filename: {output.filename}")

Output file: File ID: file-6683be733c254c78b3f11f8d9a59418e, Filename: None
Output file: File ID: file-6217c600385644ea8095b00167a68a9b, Filename: None


In [13]:
file_ids = [output.id for output in job.result.outputs if output.type == DataGenerationJobOutputType.FILE]

training_file = file_ids[0]
validation_file = file_ids[1] if len(file_ids) > 1 else None

### Inspect Output Files

Inspect the output files from the Data Generation Job. They should be ready-to-use for fine-tuning an AzureOpenAI model.

In [14]:
import json

openai_client = project_client.get_openai_client()

file_content = openai_client.files.content(training_file)
training_samples = [json.loads(line) for line in file_content.text.splitlines() if line.strip()]
training_sample = training_samples[0]
training_sample["messages"]


[{'role': 'user',
  'content': 'Hello, I want to return an item from a recent order. My email is ava.nguyen3664@example.com'},
 {'role': 'assistant',
  'content': 'null',
  'tool_calls': [{'id': 'call_je3KsA4bNjdlbxTFHWrgrtuH',
    'type': 'function',
    'function': {'name': 'find_user_id_by_email',
     'arguments': '{"email": "ava.nguyen3664@example.com"}'}}]},
 {'role': 'tool',
  'content': '{"user_id": "ava_nguyen_2175"}',
  'tool_call_id': 'call_je3KsA4bNjdlbxTFHWrgrtuH'},
 {'role': 'assistant',
  'content': 'null',
  'tool_calls': [{'id': 'call_WpEiKk7W0UBY5HDtSurEcDn9',
    'type': 'function',
    'function': {'name': 'list_user_orders',
     'arguments': '{"user_id": "ava_nguyen_2175"}'}}]},
 {'role': 'tool',
  'content': '[{"order_id": "#W1504875", "user_id": "ava_nguyen_2175", "address": {"address1": "346 Laurel Lane", "address2": "Suite 175", "city": "Austin", "country": "USA", "state": "TX", "zip": "78786"}, "items": [{"name": "Notebook", "product_id": "2892623495", "item_

In [15]:
tools = training_sample.get("tools", [])

for tool in tools:
    print(tool["function"]["name"], tool["function"].get("parameters"))

calculate {'properties': {'expression': {'type': 'string'}}, 'required': ['expression'], 'type': 'object'}
find_user_id_by_email {'properties': {'email': {'type': 'string'}}, 'required': ['email'], 'type': 'object'}
find_user_id_by_name_zip {'properties': {'first_name': {'type': 'string'}, 'last_name': {'type': 'string'}, 'zip': {'type': 'string'}}, 'required': ['first_name', 'last_name', 'zip'], 'type': 'object'}
list_all_product_types None
get_product_details {'properties': {'product_id': {'type': 'string'}}, 'required': ['product_id'], 'type': 'object'}
get_user_details {'properties': {'user_id': {'type': 'string'}}, 'required': ['user_id'], 'type': 'object'}
get_order_details {'properties': {'order_id': {'type': 'string'}}, 'required': ['order_id'], 'type': 'object'}
cancel_pending_order {'properties': {'order_id': {'type': 'string'}, 'reason': {'type': 'string'}}, 'required': ['order_id', 'reason'], 'type': 'object'}
modify_pending_order_items {'properties': {'order_id': {'type': 

### Start Fine-tuning Job

In [18]:
MODEL_NAME = "gpt-4.1-mini-2025-04-14"

finetuning_job = openai_client.fine_tuning.jobs.create(
    model=MODEL_NAME,
    training_file=training_file,
    validation_file=validation_file,
    method={
        "type": "supervised",
        "supervised": {
            "hyperparameters": {
                "n_epochs": 1,
                "batch_size": 4,
                "learning_rate_multiplier": 2,
            },
        },
    },
    suffix="agent_traces_to_sft",
    extra_body={"trainingType": "Standard"}
)

print(f"Fine-tuning job created: {finetuning_job.id}")


Fine-tuning job created: ftjob-61512e79de71461893d181221431fe9c


In [17]:
while finetuning_job.status not in ["succeeded", "failed", "cancelled"]:
    events = openai_client.fine_tuning.jobs.list_events(finetuning_job.id)
    clear_output(wait=True)
    print("Latest job events:")
    for event in events.data[-3:]:
        local_time = datetime.fromtimestamp(event.created_at).strftime("%Y-%m-%d %H:%M:%S")
        print("-", local_time, event.message)
    time.sleep(10)

    finetuning_job = openai_client.fine_tuning.jobs.retrieve(finetuning_job.id)

print("Fine-tuning finished with status:", finetuning_job.status)
if finetuning_job.status == "failed":
    raise Exception("Fine-tuning job failed with error:", finetuning_job.error)

Latest job events:
- 2026-05-20 15:01:57 Preprocessing completed for file training file.
- 2026-05-20 14:59:56 Preprocessing running for file training file.
- 2026-05-20 14:58:08 Job enqueued. Waiting for jobs ahead to complete.
Fine-tuning finished with status: cancelled


In [48]:
fine_tuned_model_id = finetuning_job.fine_tuned_model
print("Fine-tuned model ID:", fine_tuned_model_id)

Fine-tuned model ID: gpt-4.1-nano-2025-04-14.ft-be6f65f341214952b3d837a5d4f36eac


### Deploy Fine-tuned Model

In [23]:
from azure.mgmt.cognitiveservices import CognitiveServicesManagementClient
from azure.mgmt.cognitiveservices.models import Deployment, DeploymentProperties, DeploymentModel, Sku

# FOUNDRY_PROJECT_ID format:
# /subscriptions/{sub}/resourceGroups/{rg}/providers/Microsoft.CognitiveServices/accounts/{account}/projects/{project}
_parts = FOUNDRY_PROJECT_ID.split("/")
SUBSCRIPTION_ID = _parts[2]
RESOURCE_GROUP = _parts[4]
ACCOUNT_NAME = _parts[8]

DEPLOYMENT_NAME = fine_tuned_model_id[:64]

cogsvc_client = CognitiveServicesManagementClient(credential=credential, subscription_id=SUBSCRIPTION_ID)

In [ ]:
deployment = Deployment(
    properties=DeploymentProperties(model=DeploymentModel(format="OpenAI", name=fine_tuned_model_id, version="1")),
    sku=Sku(name="GlobalStandard", capacity=50),
)

print(f"Deploying fine-tuned model: {DEPLOYMENT_NAME}")
poller = cogsvc_client.deployments.begin_create_or_update(
    resource_group_name=RESOURCE_GROUP,
    account_name=ACCOUNT_NAME,
    deployment_name=DEPLOYMENT_NAME,
    deployment=deployment,
)
print(f"Model deployment {deployment.name} created.")

In [ ]:
print(f"Waiting for deployment {deployment.name} to complete...")
poller.result()

### Use Fine-tuned Model in Agent

Open `src/retail-agent-langgraph/agent.yaml` and update the `AZURE_AI_MODEL_DEPLOYMENT_NAME` environment variable with the above deployment name.

Then redeploy the agent using:
```shell
azd deploy
```

The agent should now be using the fine-tuned model.